In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)

column_types = {
    "id": "Int64",
    "osm_id": "str",
    "addr_street": "str",
    "addr_number": "str",
    "ground_floor_group": "str",
    "ground_floor_function": "str",
    "upper_floor_group": "str",
    "upper_floor_function": "str",
    "number_of_storeys": "Int64",
    "is_mixed_use": "boolean",
    "build_year_est": "Int64",
    "arch_style": "str",
    "condition": "Int64",
    "max_height": "float64",
    "is_heritage": "str",
    "source": "str",
    "confidence": "Int64",
    "date_of_checking": "str",
}

In [ ]:
data_gdynia = pd.read_csv(
    "../../data/raw/srodmiescie_raw_object_id.csv",
    sep=";",
    index_col=0,
    dtype=column_types,
    parse_dates=["date_of_checking"],
)
data_gdynia.head()

In [ ]:
data_gdynia_long = pd.read_csv(
    "../../data/processed/fixed/srodmiescie_long.csv",
    sep=";",
    index_col=0,
    dtype=column_types,
)
data_gdynia_long.head()

In [ ]:
category_map = pd.read_csv(
    "../../data/reference/category_map.csv",
    sep=";",
    index_col=0,
    dtype=column_types,
)

In [ ]:
data_long_categories = pd.merge(
    data_gdynia_long,
    category_map,
    left_on="ground_floor_function",
    right_index=True,
    indicator=True,
    how="outer",
    validate="many_to_one",
)

In [ ]:
assert len(data_long_categories.loc[(data_long_categories["_merge"] != "both")]) == 0
assert data_long_categories["category"].notna().all()

In [ ]:
data_long_categories = data_long_categories.loc[
    :, ["addr_street", "function_clean", "category"]
]
data_long_categories

In [ ]:
data_long_categories = (
    data_long_categories.groupby(["addr_street", "category"], as_index=False)
    .agg("size")
    .rename(columns={"size": "num_premise"})
    .assign(
        street_total=lambda d: d.groupby("addr_street")["num_premise"].transform("sum")
    )
    .assign(share=lambda d: d["num_premise"] / d["street_total"])
)
data_long_categories

In [ ]:
sum_shares_street = data_long_categories.groupby("addr_street").agg(
    sum_1=("share", "sum")
)


In [ ]:
assert np.isclose(sum_shares_street["sum_1"], 1).all()